# AutoPhOT example setup

End-to-end AutoPhOT run on a folder of FITS images: configure, run, and plot the resulting light curve. Each numbered section sets one group of parameters. A table lists what every key does, then a code cell applies it.

### What you have to set

Only three things are required before a run:

- `fits_dir`: where your FITS images live
- a target: `target_name` (resolved through TNS) or `target_ra` / `target_dec`
- `catalog.use_catalog`: the calibration catalog

Every other section is optional. The tables show both the value this example sets and the built-in default, so you can see which choices are deliberate and which you can leave alone. The complete parameter reference lives in `autophot/data/default_input.yml`; `list_parameters()` prints every key with its default.

### Before you start

- AutoPhOT installed (`conda install -c conda-forge -c astro-sean autophot`, or `pip install .` from a source checkout)
- FITS images carrying `TELESCOP`, `INSTRUME`, and a bandpass keyword such as `FILTER`
- SExtractor on `PATH` when running template subtraction
- MAST CasJobs credentials only if you plan to use `refcat`

## How this notebook is organized

`AutomatedPhotometry.load()` returns every config key as one nested dict, `autophot_input`. Each section overrides one group:

1. Paths and target (required)
2. Calibration catalog (`use_catalog` required, rest optional)
3. Preprocessing and photometry (optional)
4. WCS and distortion (optional)
5. Limiting magnitudes (optional)
6. Template subtraction (optional; enable it here if you want difference imaging)
7. Parallelism and verbosity (optional)
8. Run
9. Light curve and tables

Run the cells top to bottom. Sections marked optional can be skipped entirely and the defaults will apply.

## Setup

Imports and the default config. `_HAS_GPU` is detected once here; later sections use it for the SFFT backend and MaxiMask batch sizing.

In [ ]:
import glob
import os
import sys

# Only needed when running from a source checkout instead of an installed package.
# sys.path.append('/path/to/autophot')

from autophot import AutomatedPhotometry

autophot_input = AutomatedPhotometry.load()

_HAS_GPU = os.path.exists('/dev/kfd') or bool(glob.glob('/dev/nvidia*'))

## 1. Paths and target

`fits_dir` is scanned recursively for FITS images; flat and nested layouts both work (see Folder layout in the README). Outputs go to a `<fits_dir>_<outdir_name>` sibling.

| Key | Set to | Default | Purpose |
|---|---|---|---|
| `fits_dir` | `'/path/to/photometry'` | **required** | Input directory holding the FITS images |
| `target_name` | `'SN2026abc'` | **required*** | TNS-resolved name; *or set `target_ra` / `target_dec` instead |
| `wdir` | `'/path/to/autophot_db'` | `'.'` | Working directory for caches, catalog queries, and the telescope database |
| `outdir_name` | `'REDUCED'` | `'REDUCED'` | Suffix of the mirrored output directory |
| `selected_filters` | `['g', 'r', 'i']` | `[]` (all bands) | Bands to keep; an empty list processes everything |
| `restart` | `False` | `True` | `False` skips images that already produced an `Output_*.csv` |

Optional keys shown commented in the code:

- `target_ra` / `target_dec`: explicit coordinates instead of a TNS lookup (mutually exclusive with a real `target_name`)
- `additional_targets` / `additional_targets_ra_dec`: extra sources fitted alongside the primary target

In [ ]:
autophot_input['fits_dir'] = '/path/to/photometry'
autophot_input['wdir'] = '/path/to/autophot_db'
autophot_input['outdir_name'] = 'REDUCED'

autophot_input['target_name'] = 'SN2026abc'
# autophot_input['target_ra'] = 123.456789
# autophot_input['target_dec'] = -12.345678
# autophot_input['additional_targets'] = ['SN2021xyz']
# autophot_input['additional_targets_ra_dec'] = [[123.4, -12.3]]

autophot_input['selected_filters'] = ['g', 'r', 'i']
autophot_input['restart'] = False

## 2. Calibration catalog

The catalog supplies reference magnitudes for the zeropoint fit. `use_catalog` is required; the remaining keys matter only for `refcat` or `gaia_custom`.

| Key | Set to | Default | Purpose |
|---|---|---|---|
| `catalog.use_catalog` | `'auto'` | **required** | Catalog backend: one name, `'auto'`, or a band-to-catalog dict |
| `catalog.MASTcasjobs_wsid` | `'...'` | `null` | MAST workspace ID for `refcat` queries |
| `catalog.MASTcasjobs_pwd` | `'...'` | `null` | MAST token for `refcat` queries |
| `catalog.transmission_curve_map` | `{'band': path}` | `null` | Local throughput curves for `gaia_custom` bands |
| `catalog.curve_map_svo` | `{'band': id}` | `null` | SVO Filter Profile Service IDs to fetch curves from |

Notes on `use_catalog` values:

- `'auto'` measures each image footprint once and keeps the catalog with the most usable calibrators per band; the choice is cached under `<wdir>/catalog_queries/`
- Single backend names include `'gaia'`, `'panstarrs'`, `'sdss'`, `'apass'`, `'refcat'`, `'2mass'`, `'skymapper'`, `'legacy'`
- A dict maps band groups to backends, e.g. `{'grizJHK': 'refcat', 'UBVRI': 'apass', 'default': 'gaia'}`; case matters (`r` is SDSS r, `R` is Johnson-Cousins R)
- `'gaia_custom'` convolves Gaia XP spectra with your throughput curves for bands no survey covers

In [ ]:
autophot_input['catalog']['use_catalog'] = 'auto'
# autophot_input['catalog']['use_catalog'] = {
#     'grizJHK': 'refcat',
#     'u': 'gaia',
#     'UBVRI': 'apass',
#     'default': 'gaia',
# }

# autophot_input['catalog']['MASTcasjobs_wsid'] = '...'
# autophot_input['catalog']['MASTcasjobs_pwd'] = '...'

# autophot_input['catalog']['transmission_curve_map'] = {'myband': '/path/to/myband.dat'}
# autophot_input['catalog']['curve_map_svo'] = {'r': 'SDSS/SDSS.r'}

## 3. Preprocessing and photometry

Optional tuning of image cleaning and the photometric measurement. The defaults work for a first run.

| Key | Set to | Default | Purpose |
|---|---|---|---|
| `cosmic_rays.remove_cmrays` | `False` | `False` | Cosmic-ray rejection; slow on large frames |
| `preprocessing.trim_image` | `5` | `0` (no trim) | Crop to an arcmin box around the target; trims the run to the region of interest |
| `background.use_maximask` | `False` | `False` | CNN defect mask on top of the heuristic masks; needs `maximask-and-maxitrack` and tensorflow |
| `background.maximask_allow_gpu` | `_HAS_GPU` | `False` | Allow MaxiMask on GPU; far faster than CPU |
| `background.maximask_batch_size` | `32` or `4` | `4` | Inference batch size, GPU vs CPU |
| `photometry.psf_oversample` | `1` | `1` | ePSF oversampling factor |
| `photometry.perform_emcee_fitting_s2n` | `100000` | `10` | S/N below which the MCMC fitter takes over; this example's huge value runs MCMC on every detection |
| `photometry.apply_colorterms` | `False` | `False` | Apply catalog color terms in the zeropoint fit |
| `photometry.psf_spatial_grid` | `'auto'` or N | `False` | Field-varying PSF on an NxN grid for wide or distorted fields |

In [ ]:
autophot_input['cosmic_rays']['remove_cmrays'] = False
autophot_input['preprocessing']['trim_image'] = 5

autophot_input['background']['use_maximask'] = False
autophot_input['background']['maximask_allow_gpu'] = _HAS_GPU
autophot_input['background']['maximask_batch_size'] = 32 if _HAS_GPU else 4

autophot_input['photometry']['psf_oversample'] = 1
autophot_input['photometry']['perform_emcee_fitting_s2n'] = 100000
autophot_input['photometry']['apply_colorterms'] = False
# autophot_input['photometry']['psf_spatial_grid'] = 'auto'

## 4. WCS and distortion

Optional astrometry handling. `redo_wcs` defaults to on; turn it off only when every header already carries a good WCS.

| Key | Set to | Default | Purpose |
|---|---|---|---|
| `wcs.redo_wcs` | `True` | `True` | Re-solve each frame with astrometry.net |
| `wcs.solver` | `'scamp'` | `'astrometry'` | Use SCAMP + SExtractor instead of astrometry.net; needs `projection_type: 'TPV'` under SIP |
| `wcs.correct_distortion` | `False` | `False` | SCAMP + SWarp resampling for fields with real optical distortion |

- `correct_distortion` needs `scamp` and `swarp` on `PATH`; on failure the frame is processed uncorrected
- On undersampled data set `correct_distortion_resampling_type = 'LANCZOS2'`; severe distortion may warrant `scamp_distort_degrees = 4`

In [ ]:
autophot_input['wcs']['redo_wcs'] = True
# autophot_input['wcs']['solver'] = 'scamp'
autophot_input['wcs']['correct_distortion'] = False

## 5. Limiting magnitudes

Optional tuning of the injection and recovery stage that measures limiting magnitudes.

| Key | Set to | Default | Purpose |
|---|---|---|---|
| `limiting_magnitude.n_jobs` | `8` | `1` | Workers for the injection/recovery bisection |
| `limiting_magnitude.inject_final_n_sites` | `6` | `6` | Farthest-point-spaced injection sites per iteration (clamped to 6-500) |
| `limiting_magnitude.recovery_method` | `'PSF'` | `'auto'` | Photometry method used to recover injected stars; `'auto'` matches the transient's method |
| `limiting_magnitude.uncal_apply` | `True` | `True` | Rescale reported errors by the injection z-score scatter |

In [ ]:
autophot_input['limiting_magnitude']['n_jobs'] = 8
autophot_input['limiting_magnitude']['inject_final_n_sites'] = 6
# autophot_input['limiting_magnitude']['recovery_method'] = 'PSF'
# autophot_input['limiting_magnitude']['uncal_apply'] = True

## 6. Template subtraction

Optional difference imaging against a reference frame. `do_subtraction` defaults to off, so enable it here if you want subtraction at all; the other keys have sensible defaults.

| Key | Set to | Default | Purpose |
|---|---|---|---|
| `template_subtraction.do_subtraction` | `1` | `False` | Enable the subtraction stage |
| `template_subtraction.alignment_method` | `'spalipy'` | `'spalipy'` | Template registration method |
| `template_subtraction.method` | `'sfft'` | `'sfft'` | `'sfft'`, `'hotpants'`, `'zogy'`, or `'auto'` |
| `template_subtraction.sfft_backend` | `'cupy'` / `'numpy'` | `'numpy'` | GPU vs CPU SFFT backend |
| `template_subtraction.sfft_cuda_device` | `'0'` | `'0'` | CUDA device seen by each worker |
| `template_subtraction.forceconv` | `'REF'` | `'REF'` | Convolve the template to the science PSF so the science ePSF applies to the difference image |
| `template_subtraction.kernel_order` | `'0'` | `0` | Kernel polynomial degree; `'auto'` adds spatial terms only with supporting evidence |
| `template_subtraction.zogy_psf` | `'auto'` | `'auto'` | Reuse the photometry ePSF for ZOGY, else a fitted Moffat |
| `template_subtraction.sfft_memory_policy` | `'fallback'` | `'shrink'` | Response to an over-budget SFFT estimate |
| `template_subtraction.sfft_max_memory_gb` | `16` | `0` (auto) | SFFT peak-memory estimate budget in GB; `0` uses 80% of free RAM, capped at 24 |
| `template_subtraction.sfft_num_threads` | `4` | `0` (auto) | BLAS threads per SFFT run |
| `template_subtraction.match_max_sources` | `2000` | `2000` | Cap on the matched-source list used for kernel fitting |
| `template_subtraction.download_templates` | `'panstarrs'` | `False` | Fetch templates into `fits_dir/templates/` before subtracting |

- `sfft_memory_policy` values: `'shrink'` reduces the kernel to fit, `'fallback'` switches to HOTPANTS with the SFFT kernel left intact, `'ignore'` logs the estimate and runs the full kernel
- With `nGPU` worker pinning each subprocess already sees only its assigned GPU, so keep `sfft_cuda_device` at `'0'`
- Templates go under `fits_dir/templates/<band>_template/`; see Folder layout in the README for the full tree

In [ ]:
autophot_input['template_subtraction']['do_subtraction'] = 1
autophot_input['template_subtraction']['alignment_method'] = 'spalipy'
autophot_input['template_subtraction']['method'] = 'sfft'

autophot_input['template_subtraction']['sfft_backend'] = 'cupy' if _HAS_GPU else 'numpy'
autophot_input['template_subtraction']['sfft_cuda_device'] = '0'
autophot_input['template_subtraction']['forceconv'] = 'REF'
autophot_input['template_subtraction']['kernel_order'] = '0'
autophot_input['template_subtraction']['zogy_psf'] = 'auto'
autophot_input['template_subtraction']['sfft_memory_policy'] = 'fallback'

# autophot_input['template_subtraction']['sfft_max_memory_gb'] = 16
# autophot_input['template_subtraction']['sfft_num_threads'] = 4
# autophot_input['template_subtraction']['match_max_sources'] = 2000
# autophot_input['template_subtraction']['download_templates'] = 'panstarrs'

## 7. Parallelism and verbosity

Optional worker counts and console output.

| Key | Set to | Default | Purpose |
|---|---|---|---|
| `nCPU` | `1` | `1` | Images processed in parallel; `AUTOPHOT_NCPU` overrides |
| `nGPU` | `0` | `1` | GPUs the image workers are pinned across round-robin; `0` keeps the ambient device mask |
| `global_verbose_level` | `1` | `1` | Console verbosity `0` quiet to `3` debug; per-image `LOG_*.log` files record everything either way |

- Worker settings like `photometry.aperture_n_jobs` multiply `nCPU`, so keep the product at or below your core count
- `nGPU` only matters when `nCPU > 1` and `AUTOPHOT_NGPU` overrides it

In [ ]:
autophot_input['nCPU'] = 1
autophot_input['nGPU'] = 0
autophot_input['global_verbose_level'] = 1

## 8. Run

Executes the pipeline and returns the path of the collected output CSV. Pass `do_photometry=False` to collect existing outputs without reprocessing.

In [ ]:
loc = AutomatedPhotometry.run_photometry(default_input=autophot_input)
print(loc)

## 9. Light curve and tables

Post-run products from `lightcurve.py`. All optional.

| Function | Purpose |
|---|---|
| `plot_lightcurve` | Detections plus upper limits; returns the detections CSV |
| `differential_photometry` | Normalized differential light curve and residuals against a weighted reference-star ensemble, with variable members vetoed and bad epochs flagged |
| `check_detection_plots` | Sorts the per-image detection plots into folders |
| `generate_photometry_table` | ASCII photometry table |

- Light curves for `additional_targets` are written under the `sub_targets/` folder next to the main output

In [ ]:
from autophot.analysis.lightcurve import (
    plot_lightcurve,
    check_detection_plots, generate_photometry_table,
)
from autophot.photometry.differential_photometry import differential_photometry

_primary_name = autophot_input.get('target_name', 'Main target')

detections_loc = plot_lightcurve(
    loc, snr_limit=3, method='PSF', format='png', show=True,
    target_name=_primary_name,
)

try:
    differential_photometry(loc, method='PSF', target_name=_primary_name, show=True)
except Exception as e:
    print(f'[WARNING] Differential photometry failed: {e}')

check_detection_plots(detections_loc, method='AP')
generate_photometry_table(loc, snr_limit=3, method='PSF', target_name=_primary_name)